# Milestone 2: Vision CNN and Transfer Learning

This notebook uses product/review image URLs from the Milestone 0 processed splits to predict a product signal from images.

The preferred label is a reliable product category signal. Because this project focuses on the `All_Beauty` category and category labels are often sparse or single-class after sampling, the notebook automatically falls back to the same product rating-band target used in Milestone 1.

**Models:**
- Small CNN trained from scratch
- Transfer learning model using MobileNetV2

Milestone 3 is intentionally not implemented here.

## 1. Setup

The notebook keeps local execution manageable by caching at most 1,000 train images, 300 validation images, and 300 test images under `data/images/`.

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')
tf.keras.utils.set_random_seed(42)

RANDOM_SEED = 42
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 32
EPOCHS = 8
MIN_TRAIN_IMAGES = 20
LABEL_ORDER = ['low', 'medium', 'high']
SAMPLE_LIMITS = {
    'train': 1000,
    'validation': 300,
    'test': 300,
}

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
IMAGE_CACHE_DIR = PROJECT_ROOT / 'data' / 'images'
IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PATHS = {
    'train': PROCESSED_DIR / 'train.csv',
    'validation': PROCESSED_DIR / 'validation.csv',
    'test': PROCESSED_DIR / 'test.csv',
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Image cache directory: {IMAGE_CACHE_DIR}')

## 2. Load Processed Splits

Milestone 2 consumes the processed CSVs created by Milestone 0. The files remain review-level rows, so this notebook builds one image example per product where possible.

In [ ]:
missing_files = [path for path in SPLIT_PATHS.values() if not path.exists()]
if missing_files:
    missing_text = '\n'.join(str(path) for path in missing_files)
    raise FileNotFoundError(f'Missing processed split files. Run notebooks/00_eda.ipynb first:\n{missing_text}')

splits = {name: pd.read_csv(path, low_memory=False) for name, path in SPLIT_PATHS.items()}

for name, df in splits.items():
    print(f'{name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns')
    display(pd.DataFrame({'column': df.columns.tolist()}))
    display(df.head(2))

## 3. Detect Image URL Columns Safely

Image information can appear as URLs, lists, JSON-like strings, dictionaries, or empty placeholders. The helpers below recursively extract HTTP(S) URLs from any column whose name suggests image content.

In [ ]:
URL_PATTERN = re.compile(r'https?://[^\s\]})>"\']+')
IMAGE_COLUMN_TOKENS = ['image', 'img', 'picture', 'photo']


def is_null_like(value) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and np.isnan(value):
        return True
    if isinstance(value, str):
        return value.strip().lower() in {'', 'none', 'nan', 'null', 'n/a', '[]', '{}'}
    return False


def valid_url(url: str) -> bool:
    parsed = urllib.parse.urlparse(url)
    return parsed.scheme in {'http', 'https'} and bool(parsed.netloc)


def unique_preserving_order(items: list[str]) -> list[str]:
    seen = set()
    output = []
    for item in items:
        if item not in seen:
            seen.add(item)
            output.append(item)
    return output


def extract_urls(value) -> list[str]:
    if is_null_like(value):
        return []

    if isinstance(value, dict):
        urls = []
        for nested_value in value.values():
            urls.extend(extract_urls(nested_value))
        return unique_preserving_order(urls)

    if isinstance(value, (list, tuple, set)):
        urls = []
        for nested_value in value:
            urls.extend(extract_urls(nested_value))
        return unique_preserving_order(urls)

    text = str(value).strip()
    regex_urls = [url.rstrip('.,;') for url in URL_PATTERN.findall(text)]
    regex_urls = [url for url in regex_urls if valid_url(url)]
    if regex_urls:
        return unique_preserving_order(regex_urls)

    if text.startswith(('[', '{')):
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(text)
                return extract_urls(parsed)
            except Exception:
                continue

    return [text] if valid_url(text) else []


def find_image_columns(df: pd.DataFrame) -> list[str]:
    return [
        column
        for column in df.columns
        if any(token in column.lower() for token in IMAGE_COLUMN_TOKENS)
    ]


image_columns_by_split = {name: find_image_columns(df) for name, df in splits.items()}
display(pd.DataFrame({'split': list(image_columns_by_split), 'image_columns': list(image_columns_by_split.values())}))

## 4. Build Image Manifest and Labels

This step creates one candidate image URL per product. It first checks whether category labels are reliable enough for classification. If category labels are not reliable, it uses the product rating-band target from Milestone 1.

In [ ]:
def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    return next((column for column in candidates if column in df.columns), None)


def numeric_series(df: pd.DataFrame, candidates: list[str], default_value=np.nan) -> pd.Series:
    column = first_existing_column(df, candidates)
    if column is None:
        return pd.Series([default_value] * len(df), index=df.index, dtype='float64')
    return pd.to_numeric(df[column], errors='coerce')


def clean_category(value) -> str | pd.NA:
    if is_null_like(value):
        return pd.NA
    text = str(value).strip()
    if text.startswith(('[', '{')):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list) and parsed:
                flat = []
                for item in parsed:
                    if isinstance(item, list):
                        flat.extend(str(part) for part in item if not is_null_like(part))
                    elif not is_null_like(item):
                        flat.append(str(item))
                if flat:
                    text = flat[-1]
        except Exception:
            pass
    text = text.strip()
    if text.lower() in {'unknown', 'nan', 'none', 'null', 'all_beauty', 'all beauty'}:
        return pd.NA
    return text


def mode_or_na(values: pd.Series):
    cleaned = values.dropna()
    if cleaned.empty:
        return pd.NA
    return cleaned.mode().iloc[0]


def make_rating_band(rating: pd.Series) -> pd.Series:
    clipped = pd.to_numeric(rating, errors='coerce').clip(lower=1, upper=5)
    return pd.cut(
        clipped,
        bins=[0, 3, 4, 5.01],
        labels=LABEL_ORDER,
        right=False,
        include_lowest=True,
    ).astype('string')


def row_image_urls(row: pd.Series, image_columns: list[str]) -> list[str]:
    urls = []
    for column in image_columns:
        urls.extend(extract_urls(row.get(column)))
    return unique_preserving_order(urls)


def first_url_from_lists(values: pd.Series):
    for urls in values:
        for url in urls:
            if valid_url(url):
                return url
    return pd.NA


def build_image_manifest(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    if 'product_id' not in df.columns:
        raise ValueError(f'{split_name} is missing product_id. Rerun Milestone 0.')

    image_columns = image_columns_by_split.get(split_name, [])
    work = pd.DataFrame(index=df.index)
    work['product_id'] = df['product_id'].astype('string').str.strip()
    work['product_title'] = df[first_existing_column(df, ['product_title', 'title'])].fillna('Unknown') if first_existing_column(df, ['product_title', 'title']) else 'Unknown'
    work['average_rating'] = numeric_series(df, ['average_rating', 'product_average_rating'])
    work['review_rating'] = numeric_series(df, ['review_rating', 'rating', 'overall'])
    work['category_label'] = df[first_existing_column(df, ['main_category', 'product_categories'])].map(clean_category) if first_existing_column(df, ['main_category', 'product_categories']) else pd.NA
    work['image_urls'] = df.apply(lambda row: row_image_urls(row, image_columns), axis=1)
    work = work.dropna(subset=['product_id'])
    work = work[work['product_id'].str.len() > 0]
    work = work[work['image_urls'].map(bool)]

    if work.empty:
        return pd.DataFrame(columns=['product_id', 'product_title', 'image_url', 'category_label', 'target_rating', 'target_source', 'rating_band', 'split'])

    manifest = work.groupby('product_id', as_index=False).agg(
        product_title=('product_title', mode_or_na),
        image_url=('image_urls', first_url_from_lists),
        category_label=('category_label', mode_or_na),
        product_average_rating=('average_rating', 'median'),
        mean_review_rating=('review_rating', 'mean'),
    )
    manifest['split'] = split_name
    manifest['target_rating'] = manifest['product_average_rating'].where(
        manifest['product_average_rating'].notna(),
        manifest['mean_review_rating'],
    )
    manifest['target_source'] = np.where(
        manifest['product_average_rating'].notna(),
        'average_rating',
        'mean_review_rating',
    )
    manifest.loc[manifest['target_rating'].isna(), 'target_source'] = 'missing'
    manifest['rating_band'] = make_rating_band(manifest['target_rating'])
    manifest = manifest.dropna(subset=['image_url']).reset_index(drop=True)
    return manifest


manifests = {name: build_image_manifest(df, name) for name, df in splits.items()}

manifest_summary = pd.DataFrame(
    [
        {
            'split': name,
            'products_with_image_url': len(df),
            'rating_band_labels': df['rating_band'].notna().sum() if 'rating_band' in df else 0,
            'category_labels': df['category_label'].notna().sum() if 'category_label' in df else 0,
        }
        for name, df in manifests.items()
    ]
)
display(manifest_summary)
display(manifests['train'].head())

In [ ]:
def choose_label_column(train_manifest: pd.DataFrame) -> tuple[str, list[str], str]:
    category_counts = train_manifest['category_label'].dropna().astype(str).value_counts()
    category_reliable = (
        len(category_counts) >= 2
        and (category_counts >= 20).sum() >= 2
        and category_counts.iloc[0] / category_counts.sum() <= 0.85
    ) if len(category_counts) else False

    if category_reliable:
        labels = category_counts.index.tolist()
        return 'category_label', labels, 'Using product category labels because they contain multiple sufficiently represented classes.'

    rating_labels = [label for label in LABEL_ORDER if label in set(train_manifest['rating_band'].dropna().astype(str))]
    return 'rating_band', rating_labels, 'Category labels are sparse or single-class, so the notebook uses product rating bands.'


label_column, label_order, label_reason = choose_label_column(manifests['train'])
print(f'Label column: {label_column}')
print(label_reason)
print('Label order:', label_order)

for split_name, manifest in manifests.items():
    manifest['label'] = manifest[label_column].astype('string')
    manifest.dropna(subset=['label'], inplace=True)
    manifest.reset_index(drop=True, inplace=True)

display(pd.DataFrame({name: manifest['label'].value_counts() for name, manifest in manifests.items()}).fillna(0).astype(int))

## 5. Download and Cache Images

The downloader skips broken URLs and validates cached files with TensorFlow image decoding. Successfully cached images are stored under `data/images/{split}/{label}/`.

In [ ]:
def safe_slug(value: str) -> str:
    slug = re.sub(r'[^a-zA-Z0-9_-]+', '_', str(value).strip().lower()).strip('_')
    return slug or 'unknown'


def image_extension_from_url(url: str) -> str:
    suffix = Path(urllib.parse.urlparse(url).path).suffix.lower()
    return suffix if suffix in {'.jpg', '.jpeg', '.png', '.webp'} else '.jpg'


def cache_path_for(url: str, split_name: str, label: str) -> Path:
    digest = hashlib.sha1(url.encode('utf-8')).hexdigest()
    return IMAGE_CACHE_DIR / split_name / safe_slug(label) / f'{digest}{image_extension_from_url(url)}'


def is_valid_image_file(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        image_bytes = tf.io.read_file(str(path))
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        return int(tf.size(image).numpy()) > 0
    except Exception:
        return False


def download_image(url: str, output_path: Path, timeout: int = 10) -> bool:
    if output_path.exists() and is_valid_image_file(output_path):
        return True

    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(
        url,
        headers={
            'User-Agent': 'Mozilla/5.0 (compatible; SmartProductIntelligence/1.0)',
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            content_type = response.headers.get('Content-Type', '')
            payload = response.read()
        if 'image' not in content_type.lower() and len(payload) < 1024:
            return False
        output_path.write_bytes(payload)
        if is_valid_image_file(output_path):
            return True
    except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, ValueError, OSError):
        pass

    if output_path.exists():
        output_path.unlink(missing_ok=True)
    return False


def balanced_candidate_order(manifest: pd.DataFrame, random_state: int = RANDOM_SEED) -> pd.DataFrame:
    if manifest.empty:
        return manifest.copy()
    return (
        manifest.sample(frac=1, random_state=random_state)
        .sort_values('label', kind='stable')
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )


def cache_split_images(manifest: pd.DataFrame, split_name: str, max_images: int) -> pd.DataFrame:
    cached_rows = []
    attempts = 0
    candidates = balanced_candidate_order(manifest)
    start_time = time.time()

    for _, row in candidates.iterrows():
        if len(cached_rows) >= max_images:
            break
        attempts += 1
        url = str(row['image_url'])
        output_path = cache_path_for(url, split_name, row['label'])
        if download_image(url, output_path):
            cached_row = row.to_dict()
            cached_row['image_path'] = str(output_path)
            cached_rows.append(cached_row)

        if attempts % 100 == 0:
            print(f'{split_name}: {len(cached_rows):,}/{max_images:,} cached after {attempts:,} attempts')

    elapsed = time.time() - start_time
    print(f'{split_name}: cached {len(cached_rows):,} valid images from {attempts:,} attempted URLs in {elapsed:.1f}s')
    return pd.DataFrame(cached_rows)


cached_splits = {
    split_name: cache_split_images(manifest, split_name, SAMPLE_LIMITS[split_name])
    for split_name, manifest in manifests.items()
}

display(pd.DataFrame({'split': list(cached_splits), 'cached_images': [len(df) for df in cached_splits.values()]}))

## 6. Prepare TensorFlow Datasets

Only labels present in the cached training set are used. Missing or broken image files have already been filtered out.

In [ ]:
train_cached = cached_splits.get('train', pd.DataFrame()).copy()
validation_cached = cached_splits.get('validation', pd.DataFrame()).copy()
test_cached = cached_splits.get('test', pd.DataFrame()).copy()

cached_train_labels = train_cached['label'].dropna().astype(str).value_counts() if 'label' in train_cached else pd.Series(dtype=int)
active_label_order = [label for label in label_order if label in set(cached_train_labels.index)]
if len(active_label_order) < 2:
    active_label_order = cached_train_labels[cached_train_labels > 0].index.tolist()

label_to_id = {label: idx for idx, label in enumerate(active_label_order)}
id_to_label = {idx: label for label, idx in label_to_id.items()}


def finalize_cached_split(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    if df.empty or not label_to_id:
        return pd.DataFrame(columns=list(df.columns) + ['label_id'] if not df.empty else ['image_path', 'label', 'label_id'])
    output = df[df['label'].isin(label_to_id)].copy()
    output['label_id'] = output['label'].map(label_to_id).astype(int)
    output = output[output['image_path'].map(lambda path: Path(path).exists())].reset_index(drop=True)
    print(f'{split_name}: {len(output):,} usable images after label filtering')
    return output


train_ready = finalize_cached_split(train_cached, 'train')
validation_ready = finalize_cached_split(validation_cached, 'validation')
test_ready = finalize_cached_split(test_cached, 'test')

RUN_VISION_TRAINING = (
    len(active_label_order) >= 2
    and len(train_ready) >= MIN_TRAIN_IMAGES
    and len(validation_ready) > 0
    and len(test_ready) > 0
)

print('Active labels:', active_label_order)
print('Run vision training:', RUN_VISION_TRAINING)
display(pd.DataFrame({'split': ['train', 'validation', 'test'], 'images': [len(train_ready), len(validation_ready), len(test_ready)]}))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.10),
    ],
    name='data_augmentation',
)


def load_image_for_dataset(path: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label


def make_image_dataset(df: pd.DataFrame, shuffle: bool = False, augment: bool = False) -> tf.data.Dataset | None:
    if df.empty:
        return None
    paths = df['image_path'].astype(str).to_numpy()
    labels = df['label_id'].astype('int32').to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=min(len(df), 1000), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_image_for_dataset, num_parallel_calls=AUTOTUNE)
    if augment:
        dataset = dataset.map(lambda image, label: (data_augmentation(image, training=True), label), num_parallel_calls=AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


if RUN_VISION_TRAINING:
    train_ds = make_image_dataset(train_ready, shuffle=True, augment=True)
    validation_ds = make_image_dataset(validation_ready)
    test_ds = make_image_dataset(test_ready)
else:
    train_ds = validation_ds = test_ds = None
    print('Skipping model training because there are not enough cached labeled images. The notebook remains runnable after more images are available.')

## 7. Small CNN From Scratch

The scratch model is intentionally compact so it can run on CPU. It uses convolutional blocks, global pooling, dropout, and a softmax output.

In [ ]:
def build_small_cnn(num_classes: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(inputs)
    x = tf.keras.layers.Conv2D(24, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(48, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(96, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.30)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='label')(x)
    model = tf.keras.Model(inputs, outputs, name='small_cnn_from_scratch')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


small_cnn_model = None
small_cnn_history = None

if RUN_VISION_TRAINING:
    small_cnn_model = build_small_cnn(num_classes=len(active_label_order))
    small_cnn_model.summary()
    small_cnn_history = small_cnn_model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=EPOCHS,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
        verbose=1,
    )
else:
    print('Small CNN training skipped.')

## 8. Transfer Learning With MobileNetV2

The transfer model freezes a MobileNetV2 image encoder and trains a small classification head. If pretrained weights cannot be downloaded, the notebook falls back to randomly initialized weights so execution can continue.

In [ ]:
def build_mobilenet_transfer(num_classes: int) -> tf.keras.Model:
    try:
        base_model = tf.keras.applications.MobileNetV2(
            input_shape=(*IMAGE_SIZE, 3),
            include_top=False,
            weights='imagenet',
        )
        print('Loaded MobileNetV2 ImageNet weights.')
    except Exception as exc:
        print(f'Could not load ImageNet weights ({exc}). Falling back to weights=None.')
        base_model = tf.keras.applications.MobileNetV2(
            input_shape=(*IMAGE_SIZE, 3),
            include_top=False,
            weights=None,
        )

    base_model.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='label')(x)
    model = tf.keras.Model(inputs, outputs, name='mobilenetv2_transfer')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


transfer_model = None
transfer_history = None

if RUN_VISION_TRAINING:
    transfer_model = build_mobilenet_transfer(num_classes=len(active_label_order))
    transfer_model.summary()
    transfer_history = transfer_model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=EPOCHS,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
        verbose=1,
    )
else:
    print('Transfer learning training skipped.')

## 9. Training Curves

The curves compare loss and accuracy for both vision models.

In [ ]:
def plot_training_history(history, title: str) -> None:
    if history is None:
        print(f'{title}: no history to plot.')
        return
    history_df = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history_df['loss'], label='Train loss')
    axes[0].plot(history_df['val_loss'], label='Validation loss')
    axes[0].set_title(f'{title} Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].plot(history_df['accuracy'], label='Train accuracy')
    axes[1].plot(history_df['val_accuracy'], label='Validation accuracy')
    axes[1].set_title(f'{title} Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()
    plt.show()


plot_training_history(small_cnn_history, 'Small CNN')
plot_training_history(transfer_history, 'MobileNetV2 Transfer')

## 10. Evaluation: Accuracy, Macro-F1, Reports, and Confusion Matrices

Evaluation uses the held-out test image manifest created from the Milestone 0 test split.

In [ ]:
def predict_dataset(model: tf.keras.Model, dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    y_true_batches = []
    y_prob_batches = []
    for _, labels in dataset:
        y_true_batches.append(labels.numpy())
    probabilities = model.predict(dataset, verbose=0)
    y_true = np.concatenate(y_true_batches) if y_true_batches else np.array([], dtype=int)
    y_pred = probabilities.argmax(axis=1) if len(probabilities) else np.array([], dtype=int)
    return y_true, y_pred, probabilities


def model_metrics(model_name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float | str]:
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }


evaluation_results = []
prediction_outputs = {}

if RUN_VISION_TRAINING:
    for model_name, model in [('Small CNN', small_cnn_model), ('MobileNetV2 Transfer', transfer_model)]:
        y_true, y_pred, y_prob = predict_dataset(model, test_ds)
        prediction_outputs[model_name] = {'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob}
        evaluation_results.append(model_metrics(model_name, y_true, y_pred))

comparison_table = pd.DataFrame(evaluation_results)
display(comparison_table)

In [ ]:
def plot_confusion_matrix(ax, y_true: np.ndarray, y_pred: np.ndarray, title: str) -> None:
    labels = list(range(len(active_label_order)))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    sns.heatmap(
        matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=active_label_order,
        yticklabels=active_label_order,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')


if RUN_VISION_TRAINING and prediction_outputs:
    fig, axes = plt.subplots(1, len(prediction_outputs), figsize=(6 * len(prediction_outputs), 5))
    if len(prediction_outputs) == 1:
        axes = [axes]
    for ax, (model_name, outputs) in zip(axes, prediction_outputs.items()):
        plot_confusion_matrix(ax, outputs['y_true'], outputs['y_pred'], f'{model_name} Test Confusion Matrix')
    plt.tight_layout()
    plt.show()

    for model_name, outputs in prediction_outputs.items():
        print(f'{model_name} classification report:')
        report = classification_report(
            outputs['y_true'],
            outputs['y_pred'],
            labels=list(range(len(active_label_order))),
            target_names=active_label_order,
            zero_division=0,
            output_dict=True,
        )
        display(pd.DataFrame(report).T)
else:
    print('No trained model outputs available for confusion matrices or classification reports.')

## 11. Sample Predictions

Sample predictions make it easier to inspect whether the visual model is learning meaningful image cues or only class priors.

In [ ]:
def read_image_for_plot(path: str) -> np.ndarray:
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    return image.numpy().astype('uint8')


def plot_sample_predictions(model_name: str, max_samples: int = 9) -> None:
    if model_name not in prediction_outputs or test_ready.empty:
        print(f'{model_name}: no predictions available.')
        return

    outputs = prediction_outputs[model_name]
    sample_count = min(max_samples, len(test_ready), len(outputs['y_pred']))
    if sample_count == 0:
        print(f'{model_name}: no samples to plot.')
        return

    sample = test_ready.head(sample_count).copy().reset_index(drop=True)
    sample['pred_label'] = [id_to_label[int(value)] for value in outputs['y_pred'][:sample_count]]
    sample['confidence'] = outputs['y_prob'][:sample_count].max(axis=1)

    cols = 3
    rows = int(np.ceil(sample_count / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sample.iterrows()):
        ax.imshow(read_image_for_plot(row['image_path']))
        ax.axis('off')
        ax.set_title(
            f"true={row['label']}\npred={row['pred_label']} ({row['confidence']:.2f})",
            fontsize=10,
        )

    for ax in axes[sample_count:]:
        ax.axis('off')

    fig.suptitle(f'{model_name} Sample Predictions', y=1.02)
    plt.tight_layout()
    plt.show()


plot_sample_predictions('Small CNN')
plot_sample_predictions('MobileNetV2 Transfer')

## Milestone 2 Complete

This notebook completes the vision milestone:

- Safely detects image-like columns and extracts nested image URLs.
- Downloads and validates a bounded local image cache under `data/images/`.
- Builds image labels from reliable category labels or falls back to product rating bands.
- Trains a small CNN from scratch and a MobileNetV2 transfer model when enough images are available.
- Reports training curves, accuracy, macro-F1, confusion matrices, classification reports, sample predictions, and a model comparison table.

Milestone 3 is not implemented here.